[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_3_Describing_a_Column.ipynb)

# 06.3: Describing a Column

Notebook 06.2 picked out particular majors. Often you want a sense of the whole column first: what is a typical salary, and how much do salaries vary? A list of 173 numbers is too much to take in by reading it, so we use **descriptive statistics**, which are numbers that summarize a column. pandas calculates them for you, so this notebook is about what each one tells you and when it can mislead you. We will not work through the formulas.

## The data

This uses the majors data again, with `salaries`, `share_women` and a new series, `category`, which gives the broad field that each major belongs to.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
salaries = pd.Series(majors["Median"].to_list(), index=major_list)
share_women = pd.Series(majors["ShareWomen"].to_list(), index=major_list)
category = pd.Series(majors["Major_category"].to_list(), index=major_list)

## What is a typical value? Mean and median

There are two common answers to this question. The **mean** is the average: add up the values and divide by how many there are. The **median** is the middle value when the values are in order, so half are below it and half are above it.

In [ ]:
print(salaries.mean())
print(salaries.median())

The mean is about 40151 and the median is 36000. They differ because a few salaries are far above the rest, such as PETROLEUM ENGINEERING at 110000. Every value counts toward the mean, so a very large value pulls the mean up. The median only cares about which value is in the middle, so it barely moves. A small example makes this easier to see.

In [ ]:
small = pd.Series([30, 30, 40, 50, 1000])
print(small.mean())
print(small.median())

The mean of 230 is higher than four of the five values, and it does not describe any of them well. The median of 40 is much closer to a typical value. When a column has a long tail of very large values, as salaries do, the data is called **skewed**, and the median usually gives a better picture of a typical value. When the mean and the median are close together, either one is a reasonable summary.

## Smallest, largest and how many

`.min()`, `.max()` and `.count()` do what their names say. `.count()` counts values that are not missing, and we will come back to why that matters.

In [ ]:
print(salaries.min())
print(salaries.max())
print(salaries.count())

Median salaries run from 22000 to 110000 across 173 majors.

## How spread out are the values? Standard deviation and percentiles

Two columns can have the same mean and look very different, so we also want a measure of spread. The **standard deviation** describes roughly how far values typically are from the mean. A larger number means the values are more spread out, and it is in the same units as the data.

In [ ]:
salaries.std()

The standard deviation of about 11470 says that salaries are typically about $11,000 from the mean of about $40,000. Because it is based on the mean, the standard deviation is also pulled up by very large values.

A **percentile** gives the value that a certain percentage of the data falls below. The 25th percentile is the value with 25 percent of the data below it, and the 50th percentile is the same thing as the median. `.quantile()` takes the percentile as a fraction.

In [ ]:
print(salaries.quantile(0.25))
print(salaries.quantile(0.75))

A quarter of the majors have a median salary below 33000, and a quarter are above 45000, so the middle half of majors falls between 33000 and 45000. That range ignores the extreme values at either end, so unlike the standard deviation it is not affected by PETROLEUM ENGINEERING.

## All of them at once: `.describe()`

`.describe()` produces most of these numbers in one call, and it is usually the first thing to run on a numeric column.

In [ ]:
salaries.describe()

Read it from the top. `count` is 173 values, `mean` is about 40151 and `std` is about 11470. `min` and `max` are 22000 and 110000. The rows labeled 25%, 50% and 75% are the percentiles from above, so the median appears here as the 50% row. A quick check of the shape: the max of 110000 is much farther from the median of 36000 than the min of 22000 is, which points to a long tail of high salaries.

## Text columns: counting categories

A mean makes no sense for a column like `category`. For a column of categories, the useful summary is how many times each value appears, which is what `.value_counts()` gives you. It sorts from most to least common.

In [ ]:
category.value_counts().head()

Engineering is the most common category with 29 majors, followed by Education with 16. Passing `normalize=True` gives fractions instead of counts, so 29 out of 173 becomes about 0.168, or about 17 percent of the majors. `.unique()` lists each distinct value once, and `.nunique()` counts them.

In [ ]:
category.value_counts(normalize=True).head(2)

In [ ]:
category.nunique()

There are 16 categories. Here is a related trick that uses what you know about masks. Since `True` counts as 1 and `False` as 0, the mean of a boolean series is the fraction of values that are `True`.

In [ ]:
(salaries > 50000).mean()

About 0.145, or 14.5 percent of the majors, have a median above $50,000. That is the 25 majors you saw in 06.2, divided by 173.

## Missing values

Real data often has gaps. pandas marks a missing value as `NaN`, short for "not a number". Compare the length of `share_women` with its count.

In [ ]:
print(len(share_women))
print(share_women.count())

The length is 173 but the count is 172, so one value is missing. `.isna()` makes a mask that is `True` where a value is missing, so you can both count the missing values and find them.

In [ ]:
print(share_women.isna().sum())
share_women[share_women.isna()]

FOOD SCIENCE is the one missing share. The statistics in this notebook all skip missing values automatically, so `share_women.mean()` averages the 172 values that exist. That is usually what you want, but it means a summary can silently leave out rows, so it is worth comparing `count` with the length of a series. Notebook 07.3 looks at what to do about missing values.

## What's next

You can now summarize a numeric column and a column of categories. All of this treats text as whole values to count, though. Often the information you want is inside the text, such as whether a major's name includes the word ENGINEERING. Notebook 06.4 shows how to work with the text in a series.